In [1]:
from __future__ import annotations

import os
from datetime import UTC, datetime, timedelta
from getpass import getpass
from typing import Any

import httpx
import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
def fetch_espn_nba_scoreboard(date_value: datetime) -> dict[str, Any]:
    url = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard"

    params = {
        "dates": date_value.strftime("%Y%m%d"),
    }

    headers = {
        "User-Agent": "Mozilla/5.0",
    }

    with httpx.Client(
        timeout=30.0,
        follow_redirects=True,
        headers=headers,
    ) as http_client:
        response = http_client.get(url, params=params)
        response.raise_for_status()
        return response.json()

In [6]:
today = datetime.now(tz=UTC)

scoreboard = fetch_espn_nba_scoreboard(today)

print(scoreboard.keys())
print("Events:", len(scoreboard.get("events", [])))

if scoreboard.get("events"):
    print(scoreboard["events"][0].keys())
else:
    print("No NBA games for this date.")

dict_keys(['leagues', 'events', 'provider'])
Events: 1
dict_keys(['id', 'uid', 'date', 'name', 'shortName', 'season', 'competitions', 'links', 'status'])


In [7]:
def parse_espn_datetime(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

In [8]:
def get_competitor_by_side(competitors: list[dict[str, Any]], side: str) -> dict[str, Any]:
    for competitor in competitors:
        if competitor.get("homeAway") == side:
            return competitor

    raise ValueError(f"Could not find competitor side: {side}")

In [9]:
def parse_score(value: Any) -> int | None:
    if value is None:
        return None

    if value == "":
        return None

    return int(value)

In [10]:
def normalize_espn_event(event: dict[str, Any]) -> dict[str, Any]:
    competition = event["competitions"][0]
    competitors = competition["competitors"]

    home = get_competitor_by_side(competitors, "home")
    away = get_competitor_by_side(competitors, "away")

    home_team = home["team"]
    away_team = away["team"]

    home_score = parse_score(home.get("score"))
    away_score = parse_score(away.get("score"))

    status = event["status"]["type"]

    game_state = status.get("state", "unknown")
    status_name = status.get("name", "unknown")
    status_detail = status.get("detail", "unknown")
    completed = bool(status.get("completed", False))

    winner = "not completed"
    margin = None
    total_points = None

    if home_score is not None and away_score is not None:
        total_points = home_score + away_score

        if completed:
            margin = abs(home_score - away_score)

            if home_score > away_score:
                winner = home_team["displayName"]
            elif away_score > home_score:
                winner = away_team["displayName"]
            else:
                winner = "tie"

    venue = competition.get("venue", {}).get("fullName", "unknown venue")

    game_date = parse_espn_datetime(event["date"])

    if home_score is None or away_score is None:
        score_text = "score not available yet"
    else:
        score_text = f"{away_team['displayName']} {away_score} - {home_team['displayName']} {home_score}"

    summary = (
        f"NBA game on {game_date.date()}: {away_team['displayName']} played "
        f"{home_team['displayName']} at {venue}. Status: {status_detail}. "
        f"Score: {score_text}. Winner: {winner}."
    )

    if completed and margin is not None and total_points is not None:
        if margin >= 15:
            game_label = "decisive result"
        elif margin <= 5:
            game_label = "close game"
        else:
            game_label = "moderate margin game"

        if total_points >= 240:
            scoring_label = "high scoring game"
        elif total_points <= 200:
            scoring_label = "low scoring game"
        else:
            scoring_label = "normal scoring game"
    else:
        game_label = "scheduled or live game"
        scoring_label = "scoring unknown"

    return {
        "espn_game_id": event["id"],
        "game_date": game_date,
        "season_year": int(event.get("season", {}).get("year", 0)),
        "season_type": int(event.get("season", {}).get("type", 0)),
        "name": event.get("name", ""),
        "short_name": event.get("shortName", ""),
        "venue": venue,
        "home_team": home_team["displayName"],
        "home_team_abbr": home_team["abbreviation"],
        "away_team": away_team["displayName"],
        "away_team_abbr": away_team["abbreviation"],
        "home_score": home_score,
        "away_score": away_score,
        "winner": winner,
        "margin": margin,
        "total_points": total_points,
        "completed": completed,
        "game_state": game_state,
        "status_name": status_name,
        "status_detail": status_detail,
        "game_label": game_label,
        "scoring_label": scoring_label,
        "summary": summary,
    }

In [11]:
def fetch_nba_games_for_date_range(
    *,
    start_date: datetime,
    days: int,
) -> list[dict[str, Any]]:
    records = []

    for offset in range(days):
        current_date = start_date + timedelta(days=offset)

        print("Fetching:", current_date.date())

        scoreboard = fetch_espn_nba_scoreboard(current_date)

        for event in scoreboard.get("events", []):
            try:
                records.append(normalize_espn_event(event))
            except Exception as error:
                print("Failed to normalize event:", event.get("id"))
                print(error)

    return records

In [12]:
start_date = datetime.now(tz=UTC) - timedelta(days=14)

nba_game_records = fetch_nba_games_for_date_range(
    start_date=start_date,
    days=15,
)

print("Total records:", len(nba_game_records))

Fetching: 2026-05-20
Fetching: 2026-05-21
Fetching: 2026-05-22
Fetching: 2026-05-23
Fetching: 2026-05-24
Fetching: 2026-05-25
Fetching: 2026-05-26
Fetching: 2026-05-27
Fetching: 2026-05-28
Fetching: 2026-05-29
Fetching: 2026-05-30
Fetching: 2026-05-31
Fetching: 2026-06-01
Fetching: 2026-06-02
Fetching: 2026-06-03
Total records: 10


In [13]:
for record in nba_game_records[:5]:
    print(record)
    print("-" * 80)

{'espn_game_id': '401873198', 'game_date': datetime.datetime(2026, 5, 21, 0, 30, tzinfo=datetime.timezone.utc), 'season_year': 2026, 'season_type': 3, 'name': 'San Antonio Spurs at Oklahoma City Thunder', 'short_name': 'SA @ OKC', 'venue': 'Paycom Center', 'home_team': 'Oklahoma City Thunder', 'home_team_abbr': 'OKC', 'away_team': 'San Antonio Spurs', 'away_team_abbr': 'SA', 'home_score': 122, 'away_score': 113, 'winner': 'Oklahoma City Thunder', 'margin': 9, 'total_points': 235, 'completed': True, 'game_state': 'post', 'status_name': 'STATUS_FINAL', 'status_detail': 'Final', 'game_label': 'moderate margin game', 'scoring_label': 'normal scoring game', 'summary': 'NBA game on 2026-05-21: San Antonio Spurs played Oklahoma City Thunder at Paycom Center. Status: Final. Score: San Antonio Spurs 113 - Oklahoma City Thunder 122. Winner: Oklahoma City Thunder.'}
--------------------------------------------------------------------------------
{'espn_game_id': '401873342', 'game_date': datetime

In [14]:
start_date = datetime.now(tz=UTC) - timedelta(days=60)

nba_game_records = fetch_nba_games_for_date_range(
    start_date=start_date,
    days=61,
)

print("Total records:", len(nba_game_records))

Fetching: 2026-04-04
Fetching: 2026-04-05
Fetching: 2026-04-06
Fetching: 2026-04-07
Fetching: 2026-04-08
Fetching: 2026-04-09
Fetching: 2026-04-10
Fetching: 2026-04-11
Fetching: 2026-04-12
Fetching: 2026-04-13
Fetching: 2026-04-14
Fetching: 2026-04-15
Fetching: 2026-04-16
Fetching: 2026-04-17
Fetching: 2026-04-18
Fetching: 2026-04-19
Fetching: 2026-04-20
Fetching: 2026-04-21
Fetching: 2026-04-22
Fetching: 2026-04-23
Fetching: 2026-04-24
Fetching: 2026-04-25
Fetching: 2026-04-26
Fetching: 2026-04-27
Fetching: 2026-04-28
Fetching: 2026-04-29
Fetching: 2026-04-30
Fetching: 2026-05-01
Fetching: 2026-05-02
Fetching: 2026-05-03
Fetching: 2026-05-04
Fetching: 2026-05-05
Fetching: 2026-05-06
Fetching: 2026-05-07
Fetching: 2026-05-08
Fetching: 2026-05-09
Fetching: 2026-05-10
Fetching: 2026-05-11
Fetching: 2026-05-12
Fetching: 2026-05-13
Fetching: 2026-05-14
Fetching: 2026-05-15
Fetching: 2026-05-16
Fetching: 2026-05-17
Fetching: 2026-05-18
Fetching: 2026-05-19
Fetching: 2026-05-20
Fetching: 202

In [15]:
COLLECTION_NAME = "NbaGameSummary"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

games = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(name="espn_game_id", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="game_date", data_type=wvc.config.DataType.DATE),
        wvc.config.Property(name="season_year", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="season_type", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="name", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="short_name", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="venue", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="home_team", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="home_team_abbr", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="away_team", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="away_team_abbr", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="home_score", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="away_score", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="winner", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="margin", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="total_points", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="completed", data_type=wvc.config.DataType.BOOL),
        wvc.config.Property(name="game_state", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="status_name", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="status_detail", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="game_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="scoring_label", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="summary", data_type=wvc.config.DataType.TEXT),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: NbaGameSummary


In [16]:
def remove_none_values(record: dict[str, Any]) -> dict[str, Any]:
    return {
        key: value
        for key, value in record.items()
        if value is not None
    }


clean_nba_game_records = [
    remove_none_values(record)
    for record in nba_game_records
]

print("Clean records:", len(clean_nba_game_records))

Clean records: 159


In [17]:
if not clean_nba_game_records:
    raise RuntimeError(
        "No NBA game records to import. Increase the date range in the previous cell."
    )

result = games.data.insert_many(clean_nba_game_records)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [18]:
response = games.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 159


In [19]:
response = games.aggregate.over_all(total_count=True)

print("Total objects:", response.total_count)

Total objects: 159


In [20]:
response = games.query.fetch_objects(
    limit=10,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "completed",
        "game_label",
        "scoring_label",
        "summary",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print("Date:", obj.properties["game_date"])
    print("Game:", obj.properties["short_name"])
    print("Home:", obj.properties["home_team"], obj.properties.get("home_score"))
    print("Away:", obj.properties["away_team"], obj.properties.get("away_score"))
    print("Winner:", obj.properties["winner"])
    print("Completed:", obj.properties["completed"])
    print("Label:", obj.properties["game_label"])
    print("Scoring:", obj.properties["scoring_label"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

UUID: 02bd7b76-58bd-4cc6-b5fc-0e9343aabcf2
Date: 2026-04-07 23:00:00+00:00
Game: CHI @ WSH
Home: Washington Wizards 98
Away: Chicago Bulls 129
Winner: Chicago Bulls
Completed: True
Label: decisive result
Scoring: normal scoring game
Summary: NBA game on 2026-04-07: Chicago Bulls played Washington Wizards at Capital One Arena. Status: Final. Score: Chicago Bulls 129 - Washington Wizards 98. Winner: Chicago Bulls.
--------------------------------------------------------------------------------
UUID: 03a9e1b7-675c-4e04-974b-70751073fe93
Date: 2026-04-05 23:30:00+00:00
Game: LAL @ DAL
Home: Dallas Mavericks 134
Away: Los Angeles Lakers 128
Winner: Dallas Mavericks
Completed: True
Label: moderate margin game
Scoring: high scoring game
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128 - Dallas Mavericks 134. Winner: Dallas Mavericks.
------------------------------------------------------------

In [21]:
response = games.query.near_text(
    query="close NBA game decided by a small margin",
    filters=Filter.by_property("completed").equal(True),
    limit=10,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "margin",
        "total_points",
        "game_label",
        "scoring_label",
        "summary",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Game:", obj.properties["short_name"])
    print("Date:", obj.properties["game_date"])
    print("Winner:", obj.properties["winner"])
    print("Margin:", obj.properties.get("margin"))
    print("Total points:", obj.properties.get("total_points"))
    print("Label:", obj.properties["game_label"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Distance: 0.49243009090423584
Game: MIL @ BKN
Date: 2026-04-07 23:30:00+00:00
Winner: Brooklyn Nets
Margin: 6
Total points: 186
Label: moderate margin game
Summary: NBA game on 2026-04-07: Milwaukee Bucks played Brooklyn Nets at Barclays Center. Status: Final. Score: Milwaukee Bucks 90 - Brooklyn Nets 96. Winner: Brooklyn Nets.
--------------------------------------------------------------------------------
Distance: 0.4976658821105957
Game: CLE @ DET
Date: 2026-05-14 00:00:00+00:00
Winner: Cleveland Cavaliers
Margin: 4
Total points: 230
Label: close game
Summary: NBA game on 2026-05-14: Cleveland Cavaliers played Detroit Pistons at Little Caesars Arena. Status: Final/OT. Score: Cleveland Cavaliers 117 - Detroit Pistons 113. Winner: Cleveland Cavaliers.
--------------------------------------------------------------------------------
Distance: 0.49892616271972656
Game: PHX @ LAL
Date: 2026-04-11 02:30:00+00:00
Winner: Los Angeles Lakers
Margin: 28
Total points: 174
Label: decisive resul

In [22]:
response = games.query.fetch_objects(
    filters=(
        Filter.by_property("completed").equal(True)
        & Filter.by_property("total_points").greater_or_equal(230)
    ),
    limit=20,
    return_properties=[
        "game_date",
        "short_name",
        "winner",
        "home_score",
        "away_score",
        "total_points",
        "scoring_label",
        "summary",
    ],
)

for obj in response.objects:
    print("Date:", obj.properties["game_date"])
    print("Game:", obj.properties["short_name"])
    print("Winner:", obj.properties["winner"])
    print("Total points:", obj.properties["total_points"])
    print("Scoring:", obj.properties["scoring_label"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Date: 2026-04-04 19:00:00+00:00
Game: SA @ DEN
Winner: Denver Nuggets
Total points: 270
Scoring: high scoring game
Summary: NBA game on 2026-04-04: San Antonio Spurs played Denver Nuggets at Ball Arena. Status: Final/OT. Score: San Antonio Spurs 134 - Denver Nuggets 136. Winner: Denver Nuggets.
--------------------------------------------------------------------------------
Date: 2026-04-04 19:00:00+00:00
Game: WSH @ MIA
Winner: Miami Heat
Total points: 288
Scoring: high scoring game
Summary: NBA game on 2026-04-04: Washington Wizards played Miami Heat at Kaseya Center. Status: Final. Score: Washington Wizards 136 - Miami Heat 152. Winner: Miami Heat.
--------------------------------------------------------------------------------
Date: 2026-04-05 19:30:00+00:00
Game: PHX @ CHI
Winner: Phoenix Suns
Total points: 230
Scoring: normal scoring game
Summary: NBA game on 2026-04-05: Phoenix Suns played Chicago Bulls at United Center. Status: Final. Score: Phoenix Suns 120 - Chicago Bulls 110

In [23]:
team = "Los Angeles Lakers"

response = games.query.fetch_objects(
    filters=(
        Filter.by_property("home_team").equal(team)
        | Filter.by_property("away_team").equal(team)
    ),
    limit=20,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "completed",
        "summary",
    ],
)

for obj in response.objects:
    print("Date:", obj.properties["game_date"])
    print("Game:", obj.properties["short_name"])
    print("Winner:", obj.properties["winner"])
    print("Completed:", obj.properties["completed"])
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Date: 2026-04-05 23:30:00+00:00
Game: LAL @ DAL
Winner: Dallas Mavericks
Completed: True
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128 - Dallas Mavericks 134. Winner: Dallas Mavericks.
--------------------------------------------------------------------------------
Date: 2026-04-08 02:30:00+00:00
Game: OKC @ LAL
Winner: Oklahoma City Thunder
Completed: True
Summary: NBA game on 2026-04-08: Oklahoma City Thunder played Los Angeles Lakers at crypto.com Arena. Status: Final. Score: Oklahoma City Thunder 123 - Los Angeles Lakers 87. Winner: Oklahoma City Thunder.
--------------------------------------------------------------------------------
Date: 2026-04-10 02:00:00+00:00
Game: LAL @ GS
Winner: Los Angeles Lakers
Completed: True
Summary: NBA game on 2026-04-10: Los Angeles Lakers played Golden State Warriors at Chase Center. Status: Final. Score: Los Angeles Lakers 119 - Golden State W

In [24]:
response = games.query.hybrid(
    query="Lakers close game high scoring",
    alpha=0.5,
    limit=10,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "margin",
        "total_points",
        "game_label",
        "scoring_label",
        "summary",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Hybrid score:", obj.metadata.score)
    print("Game:", obj.properties["short_name"])
    print("Date:", obj.properties["game_date"])
    print("Winner:", obj.properties["winner"])
    print("Margin:", obj.properties.get("margin"))
    print("Total points:", obj.properties.get("total_points"))
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Hybrid score: 1.0
Game: LAL @ HOU
Date: 2026-04-25 00:00:00+00:00
Winner: Los Angeles Lakers
Margin: 4
Total points: 220
Summary: NBA game on 2026-04-25: Los Angeles Lakers played Houston Rockets at Toyota Center (Houston). Status: Final/OT. Score: Los Angeles Lakers 112 - Houston Rockets 108. Winner: Los Angeles Lakers.
--------------------------------------------------------------------------------
Hybrid score: 0.7962751388549805
Game: LAL @ DAL
Date: 2026-04-05 23:30:00+00:00
Winner: Dallas Mavericks
Margin: 6
Total points: 262
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128 - Dallas Mavericks 134. Winner: Dallas Mavericks.
--------------------------------------------------------------------------------
Hybrid score: 0.7673963904380798
Game: LAL @ GS
Date: 2026-04-10 02:00:00+00:00
Winner: Los Angeles Lakers
Margin: 16
Total points: 222
Summary: NBA game on 2026-04-10: Los Angeles 

In [25]:
response = games.query.hybrid(
    query="Lakers close game high scoring",
    alpha=0.5,
    limit=10,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "margin",
        "total_points",
        "game_label",
        "scoring_label",
        "summary",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Hybrid score:", obj.metadata.score)
    print("Game:", obj.properties["short_name"])
    print("Date:", obj.properties["game_date"])
    print("Winner:", obj.properties["winner"])
    print("Margin:", obj.properties.get("margin"))
    print("Total points:", obj.properties.get("total_points"))
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Hybrid score: 1.0
Game: LAL @ HOU
Date: 2026-04-25 00:00:00+00:00
Winner: Los Angeles Lakers
Margin: 4
Total points: 220
Summary: NBA game on 2026-04-25: Los Angeles Lakers played Houston Rockets at Toyota Center (Houston). Status: Final/OT. Score: Los Angeles Lakers 112 - Houston Rockets 108. Winner: Los Angeles Lakers.
--------------------------------------------------------------------------------
Hybrid score: 0.796247661113739
Game: LAL @ DAL
Date: 2026-04-05 23:30:00+00:00
Winner: Dallas Mavericks
Margin: 6
Total points: 262
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128 - Dallas Mavericks 134. Winner: Dallas Mavericks.
--------------------------------------------------------------------------------
Hybrid score: 0.7726285457611084
Game: LAL @ HOU
Date: 2026-05-02 01:30:00+00:00
Winner: Los Angeles Lakers
Margin: 20
Total points: 176
Summary: NBA game on 2026-05-02: Los Angeles 

In [26]:
response = games.query.hybrid(
    query="Lakers close game high scoring",
    alpha=0.5,
    limit=10,
    return_properties=[
        "game_date",
        "short_name",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "winner",
        "margin",
        "total_points",
        "game_label",
        "scoring_label",
        "summary",
    ],
    return_metadata=MetadataQuery(score=True),
)

for obj in response.objects:
    print("Hybrid score:", obj.metadata.score)
    print("Game:", obj.properties["short_name"])
    print("Date:", obj.properties["game_date"])
    print("Winner:", obj.properties["winner"])
    print("Margin:", obj.properties.get("margin"))
    print("Total points:", obj.properties.get("total_points"))
    print("Summary:", obj.properties["summary"])
    print("-" * 80)

Hybrid score: 1.0
Game: LAL @ HOU
Date: 2026-04-25 00:00:00+00:00
Winner: Los Angeles Lakers
Margin: 4
Total points: 220
Summary: NBA game on 2026-04-25: Los Angeles Lakers played Houston Rockets at Toyota Center (Houston). Status: Final/OT. Score: Los Angeles Lakers 112 - Houston Rockets 108. Winner: Los Angeles Lakers.
--------------------------------------------------------------------------------
Hybrid score: 0.796247661113739
Game: LAL @ DAL
Date: 2026-04-05 23:30:00+00:00
Winner: Dallas Mavericks
Margin: 6
Total points: 262
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128 - Dallas Mavericks 134. Winner: Dallas Mavericks.
--------------------------------------------------------------------------------
Hybrid score: 0.7726285457611084
Game: LAL @ HOU
Date: 2026-05-02 01:30:00+00:00
Winner: Los Angeles Lakers
Margin: 20
Total points: 176
Summary: NBA game on 2026-05-02: Los Angeles 

In [27]:
def analyze_team_games(
    team_name: str,
    *,
    limit: int = 10,
) -> None:
    filters = (
        Filter.by_property("home_team").equal(team_name)
        | Filter.by_property("away_team").equal(team_name)
    )

    response = games.generate.near_text(
        query=f"recent games for {team_name}",
        filters=filters,
        grouped_task=(
            "Analyze the retrieved NBA games for the requested team. "
            "Use only the retrieved records. "
            "Mention opponents, scores, winners, margins and game labels. "
            "Do not invent missing statistics."
        ),
        limit=limit,
        return_properties=[
            "game_date",
            "short_name",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "winner",
            "margin",
            "total_points",
            "completed",
            "game_label",
            "scoring_label",
            "summary",
        ],
    )

    print("TEAM:", team_name)
    print("-" * 80)

    print(response.generated)

    print("\nRecords used:")
    for obj in response.objects:
        print(
            "-",
            obj.properties["game_date"],
            "|",
            obj.properties["short_name"],
            "| winner:",
            obj.properties["winner"],
            "| completed:",
            obj.properties["completed"],
        )

In [28]:
analyze_team_games("Boston Celtics")

TEAM: Boston Celtics
--------------------------------------------------------------------------------
Here is the analysis of the retrieved NBA games for the Boston Celtics:

1. **Boston Celtics at Philadelphia 76ers**
   - **Date:** April 24, 2026
   - **Score:** Boston Celtics 108 - Philadelphia 76ers 100
   - **Winner:** Boston Celtics
   - **Margin:** 8
   - **Game Label:** moderate margin game

2. **Boston Celtics at Philadelphia 76ers**
   - **Date:** April 26, 2026
   - **Score:** Boston Celtics 128 - Philadelphia 76ers 96
   - **Winner:** Boston Celtics
   - **Margin:** 32
   - **Game Label:** decisive result

3. **Boston Celtics at Philadelphia 76ers**
   - **Date:** May 1, 2026
   - **Score:** Boston Celtics 93 - Philadelphia 76ers 106
   - **Winner:** Philadelphia 76ers
   - **Margin:** 13
   - **Game Label:** moderate margin game

4. **Boston Celtics at New York Knicks**
   - **Date:** April 9, 2026
   - **Score:** Boston Celtics 106 - New York Knicks 112
   - **Winner:** N

In [29]:
analyze_team_games("Los Angeles Lakers")

TEAM: Los Angeles Lakers
--------------------------------------------------------------------------------
Here is the analysis of the retrieved NBA games for the Los Angeles Lakers:

1. **Game**: Los Angeles Lakers at Houston Rockets
   - **Date**: April 25, 2026
   - **Score**: Lakers 112 - Rockets 108
   - **Winner**: Los Angeles Lakers
   - **Margin**: 4
   - **Game Label**: close game

2. **Game**: Los Angeles Lakers at Golden State Warriors
   - **Date**: April 10, 2026
   - **Score**: Lakers 119 - Warriors 103
   - **Winner**: Los Angeles Lakers
   - **Margin**: 16
   - **Game Label**: decisive result

3. **Game**: Los Angeles Lakers at Houston Rockets
   - **Date**: May 2, 2026
   - **Score**: Lakers 98 - Rockets 78
   - **Winner**: Los Angeles Lakers
   - **Margin**: 20
   - **Game Label**: decisive result

4. **Game**: Los Angeles Lakers at Houston Rockets
   - **Date**: April 27, 2026
   - **Score**: Lakers 96 - Rockets 115
   - **Winner**: Houston Rockets
   - **Margin**: 19

In [30]:
analyze_team_games("Denver Nuggets")

TEAM: Denver Nuggets
--------------------------------------------------------------------------------
Here is the analysis of the retrieved NBA games for the Denver Nuggets:

1. **Game:** Denver Nuggets at San Antonio Spurs  
   - **Date:** 2026-04-13  
   - **Score:** Denver Nuggets 128 - San Antonio Spurs 118  
   - **Winner:** Denver Nuggets  
   - **Margin:** 10  
   - **Game Label:** Moderate margin game  
   - **Scoring Label:** High scoring game  

2. **Game:** Denver Nuggets at Minnesota Timberwolves  
   - **Date:** 2026-04-24  
   - **Score:** Denver Nuggets 96 - Minnesota Timberwolves 113  
   - **Winner:** Minnesota Timberwolves  
   - **Margin:** 17  
   - **Game Label:** Decisive result  
   - **Scoring Label:** Normal scoring game  

3. **Game:** Denver Nuggets at Minnesota Timberwolves  
   - **Date:** 2026-04-26  
   - **Score:** Denver Nuggets 96 - Minnesota Timberwolves 112  
   - **Winner:** Minnesota Timberwolves  
   - **Margin:** 16  
   - **Game Label:** Decisiv

In [31]:
def find_nba_games(
    query: str,
    *,
    team_name: str | None = None,
    only_completed: bool = True,
    min_total_points: int | None = None,
    max_margin: int | None = None,
    limit: int = 10,
) -> None:
    filters = None

    if only_completed:
        filters = Filter.by_property("completed").equal(True)

    if team_name is not None:
        team_filter = (
            Filter.by_property("home_team").equal(team_name)
            | Filter.by_property("away_team").equal(team_name)
        )
        filters = team_filter if filters is None else filters & team_filter

    if min_total_points is not None:
        points_filter = Filter.by_property("total_points").greater_or_equal(min_total_points)
        filters = points_filter if filters is None else filters & points_filter

    if max_margin is not None:
        margin_filter = Filter.by_property("margin").less_or_equal(max_margin)
        filters = margin_filter if filters is None else filters & margin_filter

    response = games.query.near_text(
        query=query,
        filters=filters,
        limit=limit,
        return_properties=[
            "game_date",
            "short_name",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "winner",
            "margin",
            "total_points",
            "completed",
            "game_label",
            "scoring_label",
            "summary",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    print("QUERY:", query)
    print("TEAM:", team_name)
    print("ONLY COMPLETED:", only_completed)
    print("MIN TOTAL POINTS:", min_total_points)
    print("MAX MARGIN:", max_margin)
    print("-" * 80)

    for obj in response.objects:
        print("Distance:", obj.metadata.distance)
        print("Date:", obj.properties["game_date"])
        print("Game:", obj.properties["short_name"])
        print("Winner:", obj.properties["winner"])
        print("Margin:", obj.properties.get("margin"))
        print("Total points:", obj.properties.get("total_points"))
        print("Label:", obj.properties["game_label"])
        print("Scoring:", obj.properties["scoring_label"])
        print("Summary:", obj.properties["summary"])
        print("-" * 80)

In [32]:
find_nba_games(
    "close game decided near the end",
    max_margin=5,
)

QUERY: close game decided near the end
TEAM: None
ONLY COMPLETED: True
MIN TOTAL POINTS: None
MAX MARGIN: 5
--------------------------------------------------------------------------------
Distance: 0.591071367263794
Date: 2026-05-14 00:00:00+00:00
Game: CLE @ DET
Winner: Cleveland Cavaliers
Margin: 4
Total points: 230
Label: close game
Scoring: normal scoring game
Summary: NBA game on 2026-05-14: Cleveland Cavaliers played Detroit Pistons at Little Caesars Arena. Status: Final/OT. Score: Cleveland Cavaliers 117 - Detroit Pistons 113. Winner: Cleveland Cavaliers.
--------------------------------------------------------------------------------
Distance: 0.5921255350112915
Date: 2026-04-16 02:00:00+00:00
Game: GS @ LAC
Winner: Golden State Warriors
Margin: 5
Total points: 247
Label: close game
Scoring: high scoring game
Summary: NBA game on 2026-04-16: Golden State Warriors played LA Clippers at Intuit Dome. Status: Final. Score: Golden State Warriors 126 - LA Clippers 121. Winner: Golde

In [33]:
find_nba_games(
    "high scoring offensive game",
    min_total_points=230,
)

QUERY: high scoring offensive game
TEAM: None
ONLY COMPLETED: True
MIN TOTAL POINTS: 230
MAX MARGIN: None
--------------------------------------------------------------------------------
Distance: 0.5836519002914429
Date: 2026-04-11 02:00:00+00:00
Game: GS @ SAC
Winner: Sacramento Kings
Margin: 6
Total points: 242
Label: moderate margin game
Scoring: high scoring game
Summary: NBA game on 2026-04-11: Golden State Warriors played Sacramento Kings at Golden 1 Center. Status: Final. Score: Golden State Warriors 118 - Sacramento Kings 124. Winner: Sacramento Kings.
--------------------------------------------------------------------------------
Distance: 0.6005679368972778
Date: 2026-04-16 02:00:00+00:00
Game: GS @ LAC
Winner: Golden State Warriors
Margin: 5
Total points: 247
Label: close game
Scoring: high scoring game
Summary: NBA game on 2026-04-16: Golden State Warriors played LA Clippers at Intuit Dome. Status: Final. Score: Golden State Warriors 126 - LA Clippers 121. Winner: Golden 

In [34]:
find_nba_games(
    "Lakers close high scoring game",
    team_name="Los Angeles Lakers",
    max_margin=10,
)

QUERY: Lakers close high scoring game
TEAM: Los Angeles Lakers
ONLY COMPLETED: True
MIN TOTAL POINTS: None
MAX MARGIN: 10
--------------------------------------------------------------------------------
Distance: 0.36986905336380005
Date: 2026-04-25 00:00:00+00:00
Game: LAL @ HOU
Winner: Los Angeles Lakers
Margin: 4
Total points: 220
Label: close game
Scoring: normal scoring game
Summary: NBA game on 2026-04-25: Los Angeles Lakers played Houston Rockets at Toyota Center (Houston). Status: Final/OT. Score: Los Angeles Lakers 112 - Houston Rockets 108. Winner: Los Angeles Lakers.
--------------------------------------------------------------------------------
Distance: 0.4080730080604553
Date: 2026-04-05 23:30:00+00:00
Game: LAL @ DAL
Winner: Dallas Mavericks
Margin: 6
Total points: 262
Label: moderate margin game
Scoring: high scoring game
Summary: NBA game on 2026-04-05: Los Angeles Lakers played Dallas Mavericks at American Airlines Center. Status: Final. Score: Los Angeles Lakers 128

In [35]:
client.close()